# T07 · Descarga de la muestra TCIA/IDC y manifest.csv

Descarga **solo la muestra** (`data/sample_ids.csv`, 60 pacientes) de la colección
`fdg_pet_ct_lesions` vía **IDC** (`idc-index`, espejo TCIA *Images - Defaced*, CC BY 4.0),
convierte DICOM → NIfTI con el script oficial de lab-midas, calcula `n_lesions` y
`tmtv_ml`, genera `data/manifest.csv` y empaqueta la muestra para publicarla como
**dataset privado de Hugging Face**.

1. Panel derecho *Session options*: *Internet* **activado** (GPU no hace falta).
2. Si el PR aún no está en `main`, cambia `BRANCH` por `feat/t07-download-manifest`.
3. En *Secrets* añade `HF_TOKEN` (token de HF con permiso de escritura) y, si
   quieres, `HF_REPO_ID` (p. ej. `tu-usuario/fdg-pet-ct-lesions-t07`).
4. Ejecuta todas las celdas en orden. La descarga son ~29 GB de DICOM y los
   NIfTI otros ~30 GB: se procesa por lotes para no superar ~10–12 GB de disco.
5. La última celda sube `data/raw/` completo (~30 GB) a un dataset **privado**
   de Hugging Face e imprime el enlace. Sin token, genera un zip local.

FDAT (fuente NIfTI) está caído; ver `docs/reproducibilidad/licencia_datos.md`.

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Ronald1301/Lymphoma-Machine-Learning.git"
BRANCH = "main"  # antes del merge: "feat/t07-download-manifest"
REPO_DIR = Path("/kaggle/working/Lymphoma-Machine-Learning")

def run(*cmd: str) -> None:
    print("$", *cmd)
    subprocess.run(cmd, check=True)

if REPO_DIR.exists():
    run("git", "-C", str(REPO_DIR), "pull", "--ff-only")
else:
    run("git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR))

run(
    sys.executable, "-m", "pip", "install", "-q",
    "idc-index==0.13.5",
    "dicom2nifti==2.5.0",
    "pydicom==3.0.1",
    "nibabel==5.5.0",
    "huggingface_hub==1.33.0",  # >=1.32,<2.0: satisface datasets y transformers/gradio
    "pillow<13",  # gradio de Kaggle exige pillow<13
    "pandas", "scipy", "numpy", "nilearn", "tqdm",
)
sys.path.insert(0, str(REPO_DIR / "src"))
print("repo:", REPO_DIR)

In [ ]:
# Parche temporal para dicom2nifti 2.5.0 + numpy>=2 (Python 3.13 de Kaggle).
# upstream: https://github.com/icometrix/dicom2nifti/pull/152
import dicom2nifti.common as d2n_common

_d2n_path = Path(d2n_common.__file__)
_src = _d2n_path.read_text(encoding="utf-8")
_old = "    else:\n        # Determine required range\n        minimum_required, maximum_required = data.min(), data.max()"
_new = (
    "    else:\n"
    "        data = data.astype(numpy.int32)  # evita OverflowError con intercept negativo\n"
    "        # Determine required range\n"
    "        minimum_required, maximum_required = data.min(), data.max()"
)
if _old in _src:
    _d2n_path.write_text(_src.replace(_old, _new, 1), encoding="utf-8")
    print("parche dicom2nifti aplicado:", _d2n_path)
elif "evita OverflowError con intercept negativo" in _src:
    print("parche dicom2nifti ya aplicado")
else:
    raise RuntimeError("No se encontró el punto de parcheo en dicom2nifti.common")


## Muestra y consulta IDC

Se piden las series `PT`, `CT` y `SEG`, pero **solo del estudio baseline**:
5 pacientes tienen un segundo estudio de seguimiento en IDC que no debe descargarse.
El sufijo de `study_id` (p. ej. `..._71302`) son los últimos 5 dígitos del
`StudyInstanceUID` de DICOM, así que sirve para elegir el estudio correcto.

In [ ]:
from idc_index import IDCClient
import pandas as pd

sample = pd.read_csv(REPO_DIR / "data" / "sample_ids.csv", dtype=str)
patient_ids = sorted(sample["patient_id"].unique())
print(len(patient_ids), "pacientes en la muestra")

client = IDCClient.client()
id_list = ", ".join(f"'{p}'" for p in patient_ids)
query = f"""
SELECT
    PatientID,
    StudyInstanceUID,
    StudyDate,
    SeriesInstanceUID,
    Modality,
    SeriesDescription,
    series_size_MB
FROM index
WHERE collection_id = 'fdg_pet_ct_lesions'
  AND PatientID IN ({id_list})
  AND Modality IN ('PT', 'CT', 'SEG')
"""
all_series = client.sql_query(query)
print(len(all_series), "series en todos los estudios de esos pacientes")

# Quedarse solo con el estudio baseline de sample_ids.csv.
suffix = sample["study_id"].str.rsplit("_", n=1).str[-1]
wanted = pd.DataFrame({
    "PatientID": sample["patient_id"],
    "suffix": suffix,
    "date": sample["date"],
})
all_series["suffix"] = all_series["StudyInstanceUID"].str[-5:]
series = all_series.merge(wanted, on=["PatientID", "suffix"], how="inner")

assert len(wanted) == len(patient_ids), "study_id duplicado en la muestra"
assert series["StudyInstanceUID"].nunique() == len(patient_ids), "un estudio no mappeó"
fecha_idc = series["StudyDate"].astype(str).str.replace("-", "", regex=False)
fecha_sample = series["date"].str.replace("-", "", regex=False)
assert (fecha_idc == fecha_sample).all(), "fecha de estudio distinta"
assert series.groupby("SeriesInstanceUID").size().max() == 1, "serie duplicada"
print(len(series), "series del baseline (60 estudios x 3 modalidades)")
print(series.groupby("Modality").size().to_dict())
print("MB totales:", round(series["series_size_MB"].sum(), 1))

## Descarga y conversión por lotes

El disco de Kaggle no alcanza para los ~30 GB de DICOM **y** los ~30 GB de NIfTI
a la vez. Por eso se procesa en lotes de `BATCH_SIZE` pacientes:
descargar → convertir → validar → subir ese lote a HF → borrar el lote local.
Así nunca se ocupan más de ~10–12 GB.


In [ ]:
import os
import shutil
import re
from huggingface_hub import HfApi, upload_folder

# Elegir el filesystem con más espacio libre (scratch de Kaggle antes que /kaggle/working).
_candidates = [Path("/kaggle/temp"), Path("/tmp"), Path("/kaggle/working")]
SCRATCH_DIR = max(
    (p for p in _candidates if p.exists()),
    key=lambda p: shutil.disk_usage(p).free,
)
print("scratch:", SCRATCH_DIR, "| libre GB:", round(shutil.disk_usage(SCRATCH_DIR).free / 1e9, 1))

BATCH_SIZE = 10


def read_secret(name: str) -> str:
    """Kaggle expone los secrets por UserSecretsClient (o por variable de entorno)."""
    value = os.environ.get(name, "").strip()
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient

        return UserSecretsClient().get_secret(name).strip()
    except Exception as exc:
        print(f"No se pudo leer {name}: {exc}")
        return ""


HF_REPO_ID = read_secret("HF_REPO_ID")
HF_TOKEN = read_secret("HF_TOKEN")
print("HF_REPO_ID:", HF_REPO_ID, "| token presente:", bool(HF_TOKEN))

api = None
if HF_TOKEN and HF_REPO_ID:
    api = HfApi(token=HF_TOKEN)
    api.create_repo(repo_id=HF_REPO_ID, repo_type="dataset", private=True, exist_ok=True)
else:
    print("Sin secrets: se generan CSV pero no se sube nada a Hugging Face.")


## Conversión DICOM → NIfTI (script oficial lab-midas)

Genera `SUV.nii.gz`, `CTres.nii.gz`, `PET.nii.gz`, `CT.nii.gz` y `SEG.nii.gz`
por estudio. El script se clona una sola vez y se reutiliza en cada lote.


In [ ]:
TCIA_SCRIPTS = Path("/kaggle/working/TCIA_processing")
if not TCIA_SCRIPTS.exists():
    run("git", "clone", "--depth", "1", "--branch", "master",
        "https://github.com/lab-midas/TCIA_processing.git", str(TCIA_SCRIPTS))

OUT_DIR = Path("/kaggle/working/out")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("script de conversión:", TCIA_SCRIPTS / "tcia_dicom_to_nifti.py")

# pydicom 3.x eliminó read_file; el script de lab-midas aún lo usa.
_tcia_script = TCIA_SCRIPTS / "tcia_dicom_to_nifti.py"
_src = _tcia_script.read_text(encoding="utf-8")
if "pydicom.read_file" in _src:
    _tcia_script.write_text(
        _src.replace("pydicom.read_file", "pydicom.dcmread"), encoding="utf-8"
    )
    print("parche pydicom.dcmread aplicado:", _tcia_script)
elif "pydicom.dcmread" in _src:
    print("parche pydicom.dcmread ya aplicado")
else:
    raise RuntimeError("No se encontró pydicom.read_file/dcmread en el script de TCIA")


## Bucle: descargar → convertir → validar → subir → limpiar

Cada lote deja sus filas del manifiesto en memoria; los NIfTI se suben a `raw/`
del dataset de HF y se borran del disco local antes de seguir con el siguiente.


In [ ]:
from lymphoma.data.manifest import build_manifest, write_manifest

all_patients = sorted(sample["patient_id"].unique())
batches = [all_patients[i:i + BATCH_SIZE] for i in range(0, len(all_patients), BATCH_SIZE)]
manifest_parts = []
resumen = []

for bi, batch in enumerate(batches, start=1):
    batch_series = series[series["PatientID"].isin(batch)]
    uids = batch_series["SeriesInstanceUID"].tolist()
    dicom_dir = SCRATCH_DIR / f"dicom_{bi:02d}"
    raw_dir = SCRATCH_DIR / f"raw_{bi:02d}"

    for d in (dicom_dir, raw_dir):
        if d.exists():
            shutil.rmtree(d)
    dicom_dir.mkdir(parents=True, exist_ok=True)
    raw_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n=== lote {bi}/{len(batches)}: {len(batch)} pacientes, {len(uids)} series ===")
    client.download_dicom_series(
        seriesInstanceUID=uids,
        downloadDir=str(dicom_dir),
        dirTemplate="%PatientID/%StudyInstanceUID/%Modality_%SeriesInstanceUID",
    )

    run(
        sys.executable,
        str(TCIA_SCRIPTS / "tcia_dicom_to_nifti.py"),
        str(dicom_dir),
        str(raw_dir),
    )

    pacientes_out = sorted(p.name for p in raw_dir.iterdir() if p.is_dir())
    missing = set(batch) - set(pacientes_out)
    con_mascara = sum(
        (sd / "SEG.nii.gz").is_file()
        for pid in batch
        for sd in (raw_dir / pid).glob("*")
        if sd.is_dir()
    )
    print("convertidos:", len(pacientes_out), "| faltantes:", len(missing),
          "| con máscara:", con_mascara)
    assert not missing, f"faltan pacientes en el lote {bi}: {sorted(missing)}"
    assert con_mascara == len(batch), f"sin máscara en el lote {bi}"

    lote_manifest = build_manifest(
        raw_dir,
        sample[sample["patient_id"].isin(batch)],
        repo_root=SCRATCH_DIR,
    )
    for col in ("pet_path", "ct_path", "mask_path"):
        lote_manifest[col] = lote_manifest[col].str.replace(
            r"^raw_\d+/", "data/raw/", regex=True
        )
    manifest_parts.append(lote_manifest)
    resumen.append((bi, len(batch), len(uids), con_mascara))

    if api:
        print("subiendo NIfTI del lote", bi, "a HF...")
        upload_folder(
            folder_path=str(raw_dir),
            repo_id=HF_REPO_ID,
            repo_type="dataset",
            path_in_repo="raw",
            token=HF_TOKEN,
        )

    shutil.rmtree(dicom_dir)
    shutil.rmtree(raw_dir)
    print("lote", bi, "limpio. Libre GB:",
          round(shutil.disk_usage(SCRATCH_DIR).free / 1e9, 1))

print("resumen (lote, pacientes, series, con máscara):", resumen)


## Construir `manifest.csv` definitivo

Se concatenan los lotes y se reordenan como `sample_ids.csv`. Las rutas ya
apuntan a `data/raw/...`, tal como exige el contrato de T03.


In [ ]:
manifest = pd.concat(manifest_parts, ignore_index=True)
# Conservar el orden exacto de la muestra.
manifest = sample[["study_id"]].merge(
    manifest, on="study_id", how="left", validate="one_to_one"
)
write_manifest(manifest, OUT_DIR / "manifest.csv")
write_manifest(manifest, REPO_DIR / "data" / "manifest.csv")
print(manifest[["study_id", "diagnosis", "n_lesions", "tmtv_ml"]].head(10))
print("filas:", len(manifest), "| con máscara:", int((manifest["mask_path"] != "").sum()))


## CSV finales en Hugging Face

Los NIfTI ya se subieron por lotes a `raw/`. Aquí solo se suben `manifest.csv` y
`sample_ids.csv`, y se recuerda añadir al equipo en *Settings → Members*.


In [ ]:
shutil.copy(REPO_DIR / "data" / "sample_ids.csv", OUT_DIR / "sample_ids.csv")
print("Salidas locales en", OUT_DIR)

if api:
    for name in ("manifest.csv", "sample_ids.csv"):
        api.upload_file(
            path_or_fileobj=str(OUT_DIR / name),
            path_in_repo=name,
            repo_id=HF_REPO_ID,
            repo_type="dataset",
            token=HF_TOKEN,
        )
    print("Subido a", f"https://huggingface.co/datasets/{HF_REPO_ID}")
    print("Siguiente paso: añadir al equipo en Settings → Members y pegar el enlace en el issue #7.")
else:
    print("Sin HF_TOKEN o HF_REPO_ID: no se subió nada a Hugging Face.")
